# Lab 6: Covert Channels in Image Metadata
**Guided investigation | Student notebook**

**Scenario:** Six images contain pieces of a fictional cybersecurity incident narrative. Each image contains **one partial secret**. Your task is to find the six pieces, document how each was recovered, and reconstruct the full message in the correct order. **Do not execute any activity described by the messages.**

**Learning goals:** 
-  distinguish EXIF, PNG text, and XMP metadata; 
-  use short Python snippets to extract metadata;
-  recognize encoded text; 
-  record reproducible forensic evidence; 
-  assess the limitations of metadata-based conclusions.

**How this lab works:** First practise each small snippet on **one image**. Then select and run the relevant snippets on **each of the six images**, one at a time. You will write the final secret yourself. **No completed secret is provided in this student notebook.**

## 0. Prepare the environment
Keep this notebook beside the `Lab6` folder, which contains `Images`. Run the following two short cells.

In [ ]:
%pip -q install Pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from PIL import Image, ExifTags

image_dir = Path("Images")
print("Images found:", len(list(image_dir.glob("*.png"))))

Images found: 6


### Your evidence log
For **each image**, record the metadata type, field name, raw value (or a short excerpt), decoding method (if any), recovered message, and your interpretation. Do not assume that `getexif()` displays all metadata.

## 1. First technique: EXIF metadata (one image)
EXIF metadata can include camera details, dates, descriptions, and other tags. Try **Brain-Channel.png** first.

In [12]:
image_path = image_dir / "Activation.png"
with Image.open(image_path) as img:
    exif = img.getexif()

    print("--- Exif sub-IFD (ExifOffset) ---")
    for tag_id, value in exif.get_ifd(0x8769).items():
        name = ExifTags.TAGS.get(tag_id, tag_id)
        if isinstance(value, bytes):
            value = value.decode("utf-8", errors="replace")
        print(name, ":", value)

    print("--- GPS IFD (GPSInfo) ---")
    for tag_id, value in exif.get_ifd(0x8825).items():
        name = ExifTags.GPSTAGS.get(tag_id, tag_id)
        if isinstance(value, bytes):
            value = value.decode("utf-8", errors="replace")
        print(name, ":", value)

    print("--- Non-text img.info keys ---")
    for key, value in img.info.items():
        if not isinstance(value, str):
            print(key, ":", type(value).__name__, len(value) if hasattr(value, "__len__") else value)

--- Exif sub-IFD (ExifOffset) ---
ExifVersion : 0232
ComponentsConfiguration :  
FlashPixVersion : 0100
DateTimeOriginal : 2025:02:02 10:15:00
DateTimeDigitized : 2025:02:02 10:15:00
ColorSpace : 65535
UserComment : ASCII   VULN: Identify unpatched CVE-2024-XXXX in web gateway.
--- GPS IFD (GPSInfo) ---
GPSVersionID :   
GPSLatitudeRef : N
GPSLatitude : (40.0, 42.0, 46.08)
GPSLongitudeRef : W
GPSLongitude : (74.0, 0.0, 21.6)
--- Non-text img.info keys ---
srgb : int 0
gamma : float 0.45455
dpi : tuple 2
exif : bytes 516


**Observe:** Which EXIF field carries a message? Record it in your evidence log.

**Try it yourself:** Change `image_path` to a different image. If `EXIF tags: 0` appears, does that mean the image has no metadata? Explain.

In [4]:
# Your turn: choose another image and repeat the EXIF check.
image_path = image_dir / "Activation.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    print("EXIF tags:", len(exif))
    for tag_id, value in exif.items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)
# TODO: reuse the short EXIF snippet above

EXIF tags: 10
GPSInfo : 396
ResolutionUnit : 2
ExifOffset : 204
Make : Nikon
Model : D750
Software : Adobe Lightroom 13.0
DateTime : 2025:02:02 10:15:00
YCbCrPositioning : 1
XResolution : 72.0
YResolution : 72.0


## 2. Second technique: PNG text metadata (one image)
PNG files can contain textual information that is **not** returned by `getexif()`. Try **Brain.png**.

In [5]:
image_path = image_dir / "Brain.png"
with Image.open(image_path) as img:
    for key, value in img.info.items():
        if isinstance(value, str):
            print(key, ":", value)

Software : RECON: Perform DNS harvesting and Subdomain enumeration.


**Observe:** Which PNG text field contains a clue? Try **Chemicals.png** next using the same short snippet.

In [6]:
image_path = image_dir / "Chemicals.png"
with Image.open(image_path) as img:
    for key, value in img.info.items():
        if isinstance(value, str):
            print(key, ":", value)
# TODO: reuse the PNG text snippet above

ImageDescription : EXPLOIT: Deploy SQL Injection payload to bypass WAF.


## 3. Third technique: XMP metadata (one image)
XMP is XML-based metadata. It may be stored in PNG files even when EXIF is empty. Try **Dopamine.png**.

In [7]:
image_path = image_dir / "Dopamine.png"
with Image.open(image_path) as img:
    xmp = img.info.get("XML:com.adobe.xmp") or img.info.get("xmp")
    if isinstance(xmp, bytes):
        xmp = xmp.decode("utf-8", errors="replace")
    print(xmp if xmp else "No XMP found")

<?xpacket begin='﻿' id='W5M0MpCehiHzreSzNTczkc9d'?>
<x:xmpmeta xmlns:x='adobe:ns:meta/' x:xmptk='Image::ExifTool 12.76'>
<rdf:RDF xmlns:rdf='http://www.w3.org/1999/02/22-rdf-syntax-ns#'>

 <rdf:Description rdf:about=''
  xmlns:exif='http://ns.adobe.com/exif/1.0/'>
  <exif:MakerNote>PERSIST: Establish reverse shell via obfuscated cron job.</exif:MakerNote>
 </rdf:Description>
</rdf:RDF>
</x:xmpmeta>
<?xpacket end='r'?>


**Observe:** Locate the XML element containing the partial secret. Why did the EXIF-only approach miss it? **Note:** XMP may appear under either `XML:com.adobe.xmp` or `xmp`, depending on how the file was created and the software version.

## 4. Fourth technique: Decode suspicious text (one image)
Sometimes metadata contains **encoded** rather than readable text. Try **Music.png**. First inspect its EXIF or PNG text fields. Identify a value that resembles Base64, then decode it using this small example.

In [9]:
image_path = image_dir / "Music.png"
with Image.open(image_path) as img:
    print("--- EXIF ---")
    exif = img.getexif()
    for tag_id, value in exif.items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)
    print("--- PNG text ---")
    for key, value in img.info.items():
        if isinstance(value, str):
            print(key, ":", value)

--- EXIF ---
ResolutionUnit : 2
ExifOffset : 264
Make : Olympus
Model : OM-D E-M10
DateTime : 2025:02:06 11:11:00
YCbCrPositioning : 1
Copyright : RVhGSUw6IFR1bm5lbCBjb21wcmVzc2VkIGRhdGFiYXNlIGR1bXAgdmlhIEROUyBDb3ZlcnQgQ2hhbm5lbC4=
XResolution : 72.0
YResolution : 72.0
--- PNG text ---
Copyright : RVhGSUw6IFR1bm5lbCBjb21wcmVzc2VkIGRhdGFiYXNlIGR1bXAgdmlhIEROUyBDb3ZlcnQgQ2hhbm5lbC4=
Make : Olympus
Model : OM-D E-M10
create-date : 2025-02-06T11:11:00


**Your turn:** Replace `example` with the suspicious metadata value from **Music.png**. Do not assume every long string is Base64; use the context and validate the result.

In [10]:
encoded_message = "RVhGSUw6IFR1bm5lbCBjb21wcmVzc2VkIGRhdGFiYXNlIGR1bXAgdmlhIEROUyBDb3ZlcnQgQ2hhbm5lbC4="  # TODO: paste the encoded value found in Music.png
if encoded_message:
    print(base64.b64decode(encoded_message).decode("utf-8"))

EXFIL: Tunnel compressed database dump via DNS Covert Channel.


## 5. Your investigation: six images, one by one
Use **only the relevant short snippets** from Sections 1–4 for each image. For each image: (1) try EXIF; (2) if the clue is not visible, inspect PNG text; (3) inspect XMP if present; (4) decode only when needed; (5) document the partial secret.

| Image | Metadata type / field | Decoding needed? | Partial secret recovered |
|---|---|---|---|
| Activation.png | | | |
| Brain-Channel.png | | | |
| Brain.png | | | |
| Chemicals.png | | | |
| Dopamine.png | | | |
| Music.png | | | |

**Important:** The filename order is **not** necessarily the narrative order.

### Investigation 1: `Activation.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [14]:
image_path = image_dir / "Activation.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    print("EXIF tags:", len(exif))
    for tag_id, value in exif.items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)
    print("--- Exif sub-IFD ---")
    for tag_id, value in exif.get_ifd(0x8769).items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)
# TODO: paste or adapt the relevant short snippet from Sections 1–4

EXIF tags: 10
GPSInfo : 396
ResolutionUnit : 2
ExifOffset : 204
Make : Nikon
Model : D750
Software : Adobe Lightroom 13.0
DateTime : 2025:02:02 10:15:00
YCbCrPositioning : 1
XResolution : 72.0
YResolution : 72.0
--- Exif sub-IFD ---
ExifVersion : b'0232'
ComponentsConfiguration : b'\x01\x02\x03\x00'
FlashPixVersion : b'0100'
DateTimeOriginal : 2025:02:02 10:15:00
DateTimeDigitized : 2025:02:02 10:15:00
ColorSpace : 65535
UserComment : b'ASCII\x00\x00\x00VULN: Identify unpatched CVE-2024-XXXX in web gateway.'


### Investigation 2: `Brain-Channel.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [15]:
image_path = image_dir / "Brain-Channel.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    print("EXIF tags:", len(exif))
    for tag_id, value in exif.items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)
# TODO: paste or adapt the relevant short snippet from Sections 1–4

EXIF tags: 5
ResolutionUnit : 2
ImageDescription : LATERAL: Pivot to internal Domain Controller using Mimikatz.
YCbCrPositioning : 1
XResolution : 72.0
YResolution : 72.0


### Investigation 3: `Brain.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [16]:
image_path = image_dir / "Brain.png"
with Image.open(image_path) as img:
    for key, value in img.info.items():
        if isinstance(value, str):
            print(key, ":", value)
# TODO: paste or adapt the relevant short snippet from Sections 1–4

Software : RECON: Perform DNS harvesting and Subdomain enumeration.


### Investigation 4: `Chemicals.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [17]:
image_path = image_dir / "Chemicals.png"
with Image.open(image_path) as img:
    for key, value in img.info.items():
        if isinstance(value, str):
            print(key, ":", value)
# TODO: paste or adapt the relevant short snippet from Sections 1–4

ImageDescription : EXPLOIT: Deploy SQL Injection payload to bypass WAF.


### Investigation 5: `Dopamine.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [18]:
image_path = image_dir / "Dopamine.png"
with Image.open(image_path) as img:
    xmp = img.info.get("XML:com.adobe.xmp") or img.info.get("xmp")
    if isinstance(xmp, bytes):
        xmp = xmp.decode("utf-8", errors="replace")
    print(xmp if xmp else "No XMP found")
# TODO: paste or adapt the relevant short snippet from Sections 1–4

<?xpacket begin='﻿' id='W5M0MpCehiHzreSzNTczkc9d'?>
<x:xmpmeta xmlns:x='adobe:ns:meta/' x:xmptk='Image::ExifTool 12.76'>
<rdf:RDF xmlns:rdf='http://www.w3.org/1999/02/22-rdf-syntax-ns#'>

 <rdf:Description rdf:about=''
  xmlns:exif='http://ns.adobe.com/exif/1.0/'>
  <exif:MakerNote>PERSIST: Establish reverse shell via obfuscated cron job.</exif:MakerNote>
 </rdf:Description>
</rdf:RDF>
</x:xmpmeta>
<?xpacket end='r'?>


### Investigation 6: `Music.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [19]:
image_path = image_dir / "Music.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    print("EXIF tags:", len(exif))
    for tag_id, value in exif.items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)

encoded_message = "RVhGSUw6IFR1bm5lbCBjb21wcmVzc2VkIGRhdGFiYXNlIGR1bXAgdmlhIEROUyBDb3ZlcnQgQ2hhbm5lbC4="
if encoded_message:
    print("Decoded:", base64.b64decode(encoded_message).decode("utf-8"))
# TODO: paste or adapt the relevant short snippet from Sections 1–4

EXIF tags: 9
ResolutionUnit : 2
ExifOffset : 264
Make : Olympus
Model : OM-D E-M10
DateTime : 2025:02:06 11:11:00
YCbCrPositioning : 1
Copyright : RVhGSUw6IFR1bm5lbCBjb21wcmVzc2VkIGRhdGFiYXNlIGR1bXAgdmlhIEROUyBDb3ZlcnQgQ2hhbm5lbC4=
XResolution : 72.0
YResolution : 72.0
Decoded: EXFIL: Tunnel compressed database dump via DNS Covert Channel.


## 6. Reconstruct the complete secret
Arrange the six recovered fragments into a logical sequence. Use the **stage labels** found in the evidence to justify your order. Do **not** use image filenames as the ordering rule.

1. **Stage:** __________  **Recovered message:** __________________________________
2. **Stage:** __________  **Recovered message:** __________________________________
3. **Stage:** __________  **Recovered message:** __________________________________
4. **Stage:** __________  **Recovered message:** __________________________________
5. **Stage:** __________  **Recovered message:** __________________________________
6. **Stage:** __________  **Recovered message:** __________________________________

**Explain your reasoning:** Why is this the most plausible order?

## 7. Build an automated metadata scanner
**Learning goal:** Combine the short snippets from Parts 1–4 into a reusable tool, as you did when developing security scanners in Lab 5.

Your scanner must:
1. Loop through **all six images** automatically.
2. Extract **EXIF**, **PNG text**, and **XMP** metadata where available.
3. Record each image's filename, format, metadata field names, and any suspected hidden-message field(s).
4. Identify encoded content and explain any decoding method used; do not automatically assume all unusual text is malicious.
5. Assign a **simple, explainable risk score** and record the reason for each score.
6. Export one row per image to `metadata_report.csv`.

**Suggested classroom risk model:** +10 for a confirmed covert-message field, +5 for a suspicious encoded value requiring investigation, and +5 for an additional unusual metadata indicator supported by evidence. Avoid double-counting an encoded message as both an encoded indicator and an additional unusual indicator. These scores are **heuristics**, not forensic probabilities.

**Step 7A — Starter: loop through the six images.**

In [ ]:
for image_path in sorted(image_dir.glob("*.png")):
    with Image.open(image_path) as img:
        print(image_path.name, "→", img.format, "| fields:", len(img.info))


**Step 7B — Starter: create one result row.** Build on this structure rather than copying a complete solution.

In [ ]:
image_path = image_dir / "Brain.png"
with Image.open(image_path) as img:
    row = {
        "filename": image_path.name,
        "format": img.format,
        "metadata_fields": ", ".join(img.info.keys()),
        "risk_score": 0,
    }
print(row)

**Step 7C — Your implementation:** Complete the scanner below or create `metadata_scanner.py`.

Reuse the snippets you have already practised. Extend the result dictionary to include `suspected_field`, `encoding`, and `reason`. The scanner should record the relevant evidence and explain why a risk score was assigned. You may use `csv.DictWriter` from the starter code. **Do not hard-code the six secret messages as the detector's output.**

In [ ]:
import csv

results = []

for image_path in sorted(image_dir.glob("*.png")):
    with Image.open(image_path) as img:
        result = {
            "filename": image_path.name,
            "format": img.format,
            "metadata_fields": ", ".join(img.info.keys()),
            "suspected_field": "",    # TODO: identify relevant EXIF, PNG, or XMP field
            "encoding": "",           # TODO: indicate Base64, plain text, etc.
            "risk_score": 0,          # TODO: apply and justify your scoring rules
            "reason": "",             # TODO: explain evidence for this score
        }
        # TODO: inspect EXIF metadata
        # TODO: inspect PNG text metadata and XMP metadata
        # TODO: identify covert-message indicators and update result
        results.append(result)

with open("metadata_report.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print("Saved", len(results), "rows to metadata_report.csv")

**Reflection questions**

1. Why can an image contain a hidden message even when `getexif()` returns no tags?
2. How is **encoding** different from **encryption**? Is Base64 a secure way to hide information?
3. Why should investigators preserve the original files and record hashes?
4. Can metadata alone prove that an image was edited or that an incident occurred? What corroborating evidence would you need?
5. How does this investigation connect to the automated evidence collection and analysis you practised in Labs 4 and 5?



## Submission requirements
Submit **one PDF** named `Lab6-FirstName-LastName-StudentNumber.pdf` that includes:

- **Metadata evidence table** for all six images, identifying the metadata type, field, relevant raw value, decoding method (if applicable), and recovered fragment.
- **Screenshots of the extraction output** showing how each partial secret was found.
- **Complete reconstructed six-stage secret** in the correct order, with a brief explanation of the sequence.
- **Your completed Python metadata scanner** (code and brief explanation of its main decisions).
- **Scanner results:** `metadata_report.csv` output as a table, including the risk score and supporting reason for each image.
- **Brief interpretation:** Explain one limitation of EXIF-only extraction and one limitation of automated risk scoring.
- **Answer to reflection questions** 

Do not execute the activities described by the recovered messages.

## Grading rubric (100 points)

| Component | Points | Full-credit expectations |
|---|---:|---|
| Metadata extraction and evidence documentation | **25** | Appropriate EXIF/PNG/XMP extraction for all six images; accurate field identification; clear evidence screenshots and tables. |
| Covert-channel detection and decoding | **25** | All six hidden fragments recovered accurately; encoded content decoded correctly; extraction methods explained. |
| Six-stage secret reconstruction | **20** | Complete secret assembled in correct order with a logical justification. |
| Automated metadata scanner and CSV report | **20** | Scanner iterates through all six images, checks relevant metadata types, identifies suspected fields, exports complete CSV rows, and applies consistent, explainable risk scores. |
| Interpretations, reflections, code quality, and presentation | **10** | Readable, reproducible Python code; clear report organization; thoughtful explanation of EXIF-only and risk-scoring limitations. |
| **Total** | **100** | |